Make a table of all frames (15 frames across 8 nights)

In [1]:
import glob
from astropy.io import fits
from astropy.time import Time, TimeDelta
import astropy.units as u

rows=[]
for f in sorted(glob.glob("*.gz")):
    h = fits.getheader(f)
    t0 = Time(h['DATE-OBS'], scale='utc')
    tmid = t0 + TimeDelta(h['EXPTIME']/2, format='sec')
    rows.append((f, tmid.isot, tmid.jd))
    print(f, tmid.isot, h['EXPTIME'])

20211022181158-894-RA.wcs.proc.fits.gz 2021-10-22T18:13:58.894 240.0
20211022182027-424-RA.wcs.proc.fits.gz 2021-10-22T18:22:57.424 300.0
20211026184308-020-RA.wcs.proc.fits.gz 2021-10-26T18:45:08.020 240.0
20211026184717-057-RA.wcs.proc.fits.gz 2021-10-26T18:49:17.057 240.0
20211029182022-389-RA.wcs.proc.fits.gz 2021-10-29T18:22:22.389 240.0
20211029182431-429-RA.wcs.proc.fits.gz 2021-10-29T18:26:31.429 240.0
20211031190017-266-RA.wcs.proc.fits.gz 2021-10-31T19:02:17.266 240.0
20211031190426-339-RA.wcs.proc.fits.gz 2021-10-31T19:06:26.339 240.0
20211103182047-952-RA.wcs.proc.fits.gz 2021-11-03T18:22:47.952 240.0
20211103182457-013-RA.wcs.proc.fits.gz 2021-11-03T18:26:57.013 240.0
20211106181632-807-RA.wcs.proc.fits.gz 2021-11-06T18:18:32.807 240.0
20211106182057-122-RA.wcs.proc.fits.gz 2021-11-06T18:23:27.122 300.0
20211107180417-493-RA.wcs.proc.fits.gz 2021-11-07T18:06:17.493 240.0
20211107181245-166-RA.wcs.proc.fits.gz 2021-11-07T18:14:45.166 240.0
20211111181727-345-RA.wcs.proc.fit

Get predicted positions from JPL Horizons
Query the observatory code N50 at each mid-time

If the ID is ambiguous, try id='433P' or id='2005 QN173'. Horizons gives the prediction. Your job is to measure the actual position, and the orbit must come from your measurements, not from Horizons (your proposal says the same).

In [2]:
from astroquery.jplhorizons import Horizons
obj = Horizons(id='248370', id_type='smallbody', location='N50',
               epochs=[t_jd for _,_,t_jd in rows])
eph = obj.ephemerides()
print(eph['datetime_jd','RA','DEC','V','RA_3sigma','DEC_3sigma'])

   datetime_jd        RA      DEC      V    RA_3sigma DEC_3sigma
        d            deg      deg     mag     arcsec    arcsec  
----------------- --------- -------- ------ --------- ----------
2459510.259709421 353.84577 -2.68499 19.431     0.063      0.054
2459510.265942407  353.8453 -2.68517 19.431     0.063      0.054
2459514.281342824 353.62395 -2.78159 19.534     0.062      0.053
2459514.284225197 353.62381 -2.78165 19.534     0.062      0.053
 2459517.26553691  353.5237 -2.82564 19.609     0.061      0.052
2459517.268419317 353.52361 -2.82567 19.609     0.061      0.052
2459519.293255393 353.48694 -2.84198 19.659      0.06      0.051
2459519.296138183  353.4869 -2.84199 19.659      0.06      0.051
2459522.265832778  353.4798 -2.84601 19.731     0.059      0.051
2459522.268715428  353.4798   -2.846 19.731     0.059      0.051
2459525.262879711 353.52825 -2.82607 19.802     0.057       0.05
2459525.266286134 353.52832 -2.82603 19.802     0.057       0.05
2459526.254369132 353.556

#### Let Python compute where the asteroid should be in every frame and generate region files for you

In [2]:
import glob
import warnings
from astropy.io import fits
from astropy.wcs import WCS, FITSFixedWarning
from astropy.time import Time, TimeDelta
from astroquery.jplhorizons import Horizons

# Suppress harmless Astropy WCS header warnings
warnings.filterwarnings('ignore', category=FITSFixedWarning)

files = sorted(glob.glob("*.gz"))
tmid = []

# 1. Calculate mid-exposure Julian Dates
for f in files:
    with fits.open(f) as hdul:
        header = hdul[0].header
        t = Time(header['DATE-OBS'], scale='utc') + TimeDelta(header['EXPTIME'] / 2.0, format='sec')
        tmid.append(t.jd)

# 2. Query JPL Horizons for target 248370 at observatory location N50
eph = Horizons(id='248370', id_type='smallbody', location='N50', epochs=tmid).ephemerides()

# 3. Project celestial coordinates (RA/DEC) to pixel coordinates (X/Y) & write DS9 regions
for f, row in zip(files, eph):
    with fits.open(f) as hdul:
        w = WCS(hdul[0].header)
    
    # Transform RA/DEC to 1-based pixel coordinates for DS9
    x, y = w.all_world2pix(row['RA'], row['DEC'], 1)
    
    print(f"{f}  predicted x={float(x):.1f} y={float(y):.1f}  V={row['V']:.2f}", flush=True)
    
    # Save region file
    reg_filename = f.replace('.gz', '') + ".reg"
    with open(reg_filename, "w") as r:
        r.write("image\n")
        r.write(f"circle({float(x):.1f},{float(y):.1f},12) # color=red\n")

20211022181158-894-RA.wcs.proc.fits.gz  predicted x=2004.7 y=1840.3  V=19.43
20211022182027-424-RA.wcs.proc.fits.gz  predicted x=2005.7 y=1839.9  V=19.43
20211026184308-020-RA.wcs.proc.fits.gz  predicted x=1867.1 y=1206.0  V=19.53
20211026184717-057-RA.wcs.proc.fits.gz  predicted x=1867.0 y=1207.0  V=19.53
20211029182022-389-RA.wcs.proc.fits.gz  predicted x=2437.5 y=1206.1  V=19.61
20211029182431-429-RA.wcs.proc.fits.gz  predicted x=2436.7 y=1207.6  V=19.61
20211031190017-266-RA.wcs.proc.fits.gz  predicted x=2169.0 y=1065.3  V=19.66
20211031190426-339-RA.wcs.proc.fits.gz  predicted x=2166.3 y=1065.8  V=19.66
20211103182047-952-RA.wcs.proc.fits.gz  predicted x=2075.2 y=2030.9  V=19.73
20211103182457-013-RA.wcs.proc.fits.gz  predicted x=2076.8 y=2030.7  V=19.73
20211106181632-807-RA.wcs.proc.fits.gz  predicted x=2039.7 y=1924.0  V=19.80
20211106182057-122-RA.wcs.proc.fits.gz  predicted x=2041.4 y=1924.0  V=19.80
20211107180417-493-RA.wcs.proc.fits.gz  predicted x=2060.5 y=1918.8  V=19.82

In [1]:
import glob
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.gaia import Gaia

Gaia.ROW_LIMIT = -1          # don't truncate at 2000 rows

for f in sorted(glob.glob("*.gz")):
    h = fits.getheader(f)
    w = WCS(h)
    nx, ny = h['NAXIS1'], h['NAXIS2']

    # field centre in sky coordinates (pixels are 1-indexed here)
    ra0, dec0 = w.all_pix2world(nx/2, ny/2, 1)
    c = SkyCoord(float(ra0)*u.deg, float(dec0)*u.deg, frame='icrs')

    r = Gaia.cone_search_async(c, radius=0.42*u.deg).get_results()
    r = r[r['phot_g_mean_mag'] < 20.5]

    x, y = w.all_world2pix(np.array(r['ra']), np.array(r['dec']), 1)
    ok = (x > 1) & (x < nx) & (y > 1) & (y < ny)

    out = f.replace('.gz', '') + "_gaia.reg"
    with open(out, "w") as g:
        g.write("global color=green\nimage\n")
        for xi, yi in zip(x[ok], y[ok]):
            g.write(f"circle({xi:.2f},{yi:.2f},7)\n")
    print(f, "->", out, ok.sum(), "stars")

The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.


INFO: Query finished. [astroquery.utils.tap.core]
20211022181158-894-RA.wcs.proc.fits.gz -> 20211022181158-894-RA.wcs.proc.fits_gaia.reg 1353 stars


INFO: Query finished. [astroquery.utils.tap.core]
20211022182027-424-RA.wcs.proc.fits.gz -> 20211022182027-424-RA.wcs.proc.fits_gaia.reg 1350 stars


INFO: Query finished. [astroquery.utils.tap.core]
20211026184308-020-RA.wcs.proc.fits.gz -> 20211026184308-020-RA.wcs.proc.fits_gaia.reg 1382 stars


INFO: Query finished. [astroquery.utils.tap.core]
20211026184717-057-RA.wcs.proc.fits.gz -> 20211026184717-057-RA.wcs.proc.fits_gaia.reg 1382 stars


INFO: Query finished. [astroquery.utils.tap.core]
20211029182022-389-RA.wcs.proc.fits.gz -> 20211029182022-389-RA.wcs.proc.fits_gaia.reg 1375 stars


... (6 outputs left out)

INFO: Query finished. [astroquery.utils.tap.core]
20211106182057-122-RA.wcs.proc.fits.gz -> 20211106182057-122-RA.wcs.proc.fits_gaia.reg 1351 stars


INFO: Query finished. [astroquery.utils.tap.core]
20211107180417-493-RA.wcs.proc.fits.gz -> 20211107180417-493-RA.wcs.proc.fits_gaia.reg 1368 stars


INFO: Query finished. [astroquery.utils.tap.core]
20211107181245-166-RA.wcs.proc.fits.gz -> 20211107181245-166-RA.wcs.proc.fits_gaia.reg 1366 stars


INFO: Query finished. [astroquery.utils.tap.core]
20211111181727-345-RA.wcs.proc.fits.gz -> 20211111181727-345-RA.wcs.proc.fits_gaia.reg 1373 stars


## 01 Astrometry

In [5]:
"""
Step 1: refine the DS9 positions, convert to RA/Dec with each frame's WCS,
estimate uncertainties, and write (a) astrometry_table.csv and (b) mpc_obs.txt.

Run from the folder that contains the 15 *.gz FITS files and asteroid_xy.csv.
"""
import glob, warnings
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter
from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord
from astropy.stats import sigma_clipped_stats
import astropy.units as u

warnings.filterwarnings('ignore')

CSV        = "asteroid_xy.csv"
USE_GAIA   = True      # reference-star check (needs internet); set False to skip
PACKED_ID  = "O8370"   # (248370): 24 -> 'O', then 8370
FWHM_ARCSEC = 2.5      # rough seeing, used in the centroid-error estimate
MAX_SHIFT_PX = 4.0     # reject refinement if it wanders further than this from your click

# ---------------------------------------------------------------- centroiding
def refine_centroid(data, x_ds9, y_ds9, rad=4.0, box=20, max_shift=MAX_SHIFT_PX, sm=1.5):
    """Inputs/outputs are 1-indexed (DS9) pixels. Returns x, y, shift_px, snr."""
    x0, y0 = x_ds9 - 1.0, y_ds9 - 1.0
    ix, iy = int(round(x0)), int(round(y0))
    sub = data[iy-box:iy+box+1, ix-box:ix+box+1]
    yy, xx = np.mgrid[iy-box:iy+box+1, ix-box:ix+box+1]
    r = np.hypot(xx-x0, yy-y0)
    _, bkg, noise = sigma_clipped_stats(sub[(r > 10) & (r < box)], sigma=3)
    s = gaussian_filter(sub - bkg, sm)
    k = np.argmax(np.where(r <= max_shift, s, -np.inf))      # brightest smoothed pixel near click
    cx, cy = xx.flat[k], yy.flat[k]
    for _ in range(15):                                      # flux-weighted centroid
        m = np.hypot(xx-cx, yy-cy) <= rad
        w = np.clip(sub - bkg, 0, None) * m
        if w.sum() <= 0: break
        ncx, ncy = (w*xx).sum()/w.sum(), (w*yy).sum()/w.sum()
        done = np.hypot(ncx-cx, ncy-cy) < 1e-3
        cx, cy = ncx, ncy
        if done: break
    m = np.hypot(xx-cx, yy-cy) <= rad
    snr = ((sub-bkg)*m).sum() / (noise*np.sqrt(m.sum()))
    return cx+1, cy+1, np.hypot(cx-x0, cy-y0), snr

# ---------------------------------------------------------------- Gaia check
def reference_star_rms(data, wcs, tmid, ra0, dec0, nmax=40):
    """Centroid isolated Gaia stars and compare with their Gaia positions (arcsec)."""
    from astroquery.gaia import Gaia
    Gaia.ROW_LIMIT = -1
    c = SkyCoord(ra0*u.deg, dec0*u.deg)
    t = Gaia.cone_search_async(c, radius=0.12*u.deg).get_results()
    t = t[(t['phot_g_mean_mag'] > 14.5) & (t['phot_g_mean_mag'] < 18)]
    pmra = np.nan_to_num(np.array(t['pmra'], dtype=float)); pmde = np.nan_to_num(np.array(t['pmdec'], dtype=float))
    g = SkyCoord(np.array(t['ra'])*u.deg, np.array(t['dec'])*u.deg,
                 pm_ra_cosdec=pmra*u.mas/u.yr, pm_dec=pmde*u.mas/u.yr,
                 obstime=Time(2016.0, format='jyear')).apply_space_motion(new_obstime=tmid)
    # keep isolated stars only
    allc = SkyCoord(np.array(t['ra'])*u.deg, np.array(t['dec'])*u.deg)
    idx, sep, _ = allc.match_to_catalog_sky(allc, nthneighbor=2)
    iso = sep.arcsec > 8
    xs, ys = wcs.all_world2pix(g.ra.deg, g.dec.deg, 1)
    dra, ddec = [], []
    for x, y, ok in zip(xs, ys, iso):
        if not ok or not (40 < x < data.shape[1]-40 and 40 < y < data.shape[0]-40): continue
        try:
            cx, cy, sh, snr = refine_centroid(data, x, y, max_shift=3.0)
        except Exception:
            continue
        if sh > 2.5 or snr < 15: continue
        ra_m, de_m = wcs.all_pix2world(cx, cy, 1)
        sp = SkyCoord(ra_m*u.deg, de_m*u.deg)
        gi = SkyCoord(*wcs.all_pix2world(x, y, 1)*u.deg)
        dra.append(((ra_m - gi.ra.deg)*np.cos(np.radians(de_m)))*3600)
        ddec.append((de_m - gi.dec.deg)*3600)
        if len(dra) >= nmax: break
    if len(dra) < 5: return np.nan, len(dra)
    d = np.hypot(np.array(dra), np.array(ddec))
    # robust 1-D scatter per axis
    sc = 1.4826*np.median(np.abs(np.concatenate([dra, ddec]) - np.median(np.concatenate([dra, ddec]))))
    return sc, len(dra)

# ---------------------------------------------------------------- MPC format
def mpc_line(packed, tmid_utc, ra_deg, dec_deg, band="r"):
    d = tmid_utc.utc.datetime
    day = d.day + (d.hour + d.minute/60 + (d.second + d.microsecond/1e6)/3600)/24
    date = f"{d.year:04d} {d.month:02d} {day:09.6f}"                    # cols 16-32
    c = SkyCoord(ra_deg*u.deg, dec_deg*u.deg)
    ra = c.ra.to_string(unit=u.hourangle, sep=' ', precision=3, pad=True)   # 12 chars
    de = c.dec.to_string(sep=' ', precision=2, alwayssign=True, pad=True)   # 12 chars
    line = f"{packed:5s}{'':7s} " + " C" + date + ra + de + f"{'':9s}{'':5s}{band}{'':6s}N50"
    # cols: 1-5 number | 6-12 blank | 13 blank | 14 blank | 15 'C' | 16-32 date | 33-44 RA | 45-56 Dec
    assert len(line) == 80, len(line)
    return line

# ---------------------------------------------------------------- main
df = pd.read_csv(CSV, dtype={'ObsFile': str, 'Date': str})
rows, mpc = [], []
for _, r in df.iterrows():
    # dd, mm = r['Date'].split('/')
    # pat = f"2021{mm}{dd}*-{r['ObsFile']}-*"
    # fl = glob.glob(pat)
    dd, mm = r['Date'].split('/')
    pat = f"2021{mm.zfill(2)}{dd.zfill(2)}*-{r['ObsFile']}-*.fits.gz"
    fl = glob.glob(pat)
    assert len(fl) == 1, f"pattern {pat} matched {fl}"
    f = fl[0]
    with fits.open(f) as hdul:
        hdr, data = hdul[0].header, hdul[0].data.astype(float)
    w = WCS(hdr)
    tmid = Time(hdr['DATE-OBS'], scale='utc') + TimeDelta(hdr['EXPTIME']/2, format='sec')

    x, y, shift, snr = refine_centroid(data, r['x'], r['y'])
    flag = ""
    ra, dec = (float(v) for v in w.all_pix2world(x, y, 1))
    if USE_GAIA:
        rms_ref, nref = reference_star_rms(data, w, tmid, ra, dec)
    else:
        rms_ref, nref = np.nan, 0
    sig_cen = FWHM_ARCSEC / max(snr, 1.0)                          # centroid error ~ FWHM / SNR
    sig = np.sqrt((0 if np.isnan(rms_ref) else rms_ref)**2 + sig_cen**2)
    sig = max(sig, 0.3)
    ra_csv, dec_csv = r['RA'], r['DEC']
    diff = np.hypot((ra-ra_csv)*np.cos(np.radians(dec)), dec-dec_csv)*3600
    rows.append(dict(file=f, obs=r['ObsFile'], date_obs=hdr['DATE-OBS'], tmid_utc=tmid.isot, jd_mid=tmid.jd,
                     x_ds9=r['x'], y_ds9=r['y'], x=x, y=y, shift_px=shift, snr=snr,
                     ra_deg=ra, dec_deg=dec, ref_rms_arcsec=rms_ref, n_ref=nref,
                     sigma_arcsec=sig, diff_vs_ds9_arcsec=diff))
    mpc.append(mpc_line(PACKED_ID, tmid, ra, dec))
    print(f"{f[:22]}  tmid={tmid.isot}  xy=({x:.2f},{y:.2f})  shift={shift:.2f}px  S/N={snr:.1f}  "
          f"RA={ra:.6f} Dec={dec:.6f}  sigma={sig:.2f}\"")

out = pd.DataFrame(rows)
out.to_csv("astrometry_table.csv", index=False)
open("mpc_obs.txt", "w").write("\n".join(mpc) + "\n")
print("\nwrote astrometry_table.csv and mpc_obs.txt")
print(out[['obs','tmid_utc','x','y','shift_px','snr','sigma_arcsec']].to_string())

The Gaia ESA Archive will be down as of 25 November 12:00 CET for preparation of the Gaia DR4 release on 2 December 12:00 CET. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.
INFO: Query finished. [astroquery.utils.tap.core]
20211022181158-894-RA.  tmid=2021-10-22T18:13:58.894  xy=(2004.24,1840.53)  shift=0.42px  S/N=7.9  RA=353.845808 Dec=-2.685091  sigma=0.34"
INFO: Query finished. [astroquery.utils.tap.core]
20211022182027-424-RA.  tmid=2021-10-22T18:22:57.424  xy=(2005.61,1840.37)  shift=0.50px  S/N=8.7  RA=353.845394 Dec=-2.685206  sigma=0.30"
INFO: Query finished. [astroquery.utils.tap.core]
20211026184308-020-RA.  tmid=2021-10-26T18:45:08.020  xy=(1867.26,1205.76)  shift=0.59px  S/N=13.4  RA=353.623912 Dec=-2.781544  sigma=0.30"
INFO: Query finished. [astroquery.utils.tap.core]
20211026184717-057-RA.  tmid=2021-10-26T18:49:17.057  xy=(1867.34,1207.01)  shift=0.34px  S/N=13.5  RA=353.623826 Dec=-2.781591  si

## 002 Orbit lib

In [6]:

from scipy.optimize import least_squares
from astropy.coordinates import (get_body_barycentric, EarthLocation, SkyCoord, solar_system_ephemeris)

warnings.filterwarnings('ignore')
solar_system_ephemeris.set('builtin')
K = 0.01720209895            # Gaussian grav. constant, AU^1.5/day
MU = K**2
C_AU_DAY = 173.1446327       # speed of light
EPS = np.radians(23.4392911) # obliquity J2000
SITE = EarthLocation(lon=78.9641*u.deg, lat=32.7794*u.deg, height=4500*u.m)     # GIT's location

def observer_helio(jd):
    """Heliocentric ICRS(equatorial) position of the telescope [AU]."""
    t = Time(jd, format='jd', scale='utc')
    e = get_body_barycentric('earth', t); s = get_body_barycentric('sun', t)
    site = SITE.get_gcrs_posvel(t)[0]
    r = (e - s).xyz.to(u.au).value + site.xyz.to(u.au).value
    return r.T if r.ndim > 1 else r

def stumpff(z):
    if abs(z) < 1e-8: return 1/2, 1/6
    if z > 0: s = np.sqrt(z); return (1-np.cos(s))/z, (s-np.sin(s))/s**3
    s = np.sqrt(-z); return (np.cosh(s)-1)/(-z), (np.sinh(s)-s)/s**3

def kepler_prop(r0, v0, dt):
    """Universal-variable two-body propagation."""
    r0n = np.linalg.norm(r0); vr0 = r0@v0/r0n; alpha = 2/r0n - v0@v0/MU
    chi = np.sqrt(MU)*abs(alpha)*dt if abs(alpha) > 1e-12 else np.sqrt(MU)*dt/r0n
    for _ in range(100):
        z = alpha*chi**2; C, S = stumpff(z)
        F = r0n*vr0/np.sqrt(MU)*chi**2*C + (1-alpha*r0n)*chi**3*S + r0n*chi - np.sqrt(MU)*dt
        dF = r0n*vr0/np.sqrt(MU)*chi*(1-z*S) + (1-alpha*r0n)*chi**2*C + r0n
        d = F/dF; chi -= d
        if abs(d) < 1e-12: break
    z = alpha*chi**2; C, S = stumpff(z)
    f = 1 - chi**2/r0n*C; g = dt - chi**3/np.sqrt(MU)*S
    r = f*r0 + g*v0; rn = np.linalg.norm(r)
    fd = np.sqrt(MU)/(rn*r0n)*(z*S-1)*chi; gd = 1 - chi**2/rn*C
    return r, fd*r0 + gd*v0

def radec_model(p, jd_obs, jd_ref, Robs):
    r0, v0 = p[:3], p[3:]
    out = []
    for jd, R in zip(jd_obs, Robs):
        tau = 0.0
        for _ in range(3):                               # light-time iteration
            r, _ = kepler_prop(r0, v0, (jd - tau) - jd_ref)
            rho = r - R; tau = np.linalg.norm(rho)/C_AU_DAY
        ra = np.arctan2(rho[1], rho[0]) % (2*np.pi)
        dec = np.arcsin(rho[2]/np.linalg.norm(rho))
        out.append((ra, dec))
    return np.array(out)

def residuals(p, jd_obs, jd_ref, Robs, ra, dec, sig):
    m = radec_model(p, jd_obs, jd_ref, Robs)
    dra = ((m[:,0]-ra + np.pi) % (2*np.pi) - np.pi)*np.cos(dec)
    ddec = m[:,1]-dec
    return np.concatenate([dra/sig, ddec/sig])

def elements(p, jd_ref):
    """State (equatorial) -> ecliptic J2000 Keplerian elements."""
    R = np.array([[1,0,0],[0,np.cos(EPS),np.sin(EPS)],[0,-np.sin(EPS),np.cos(EPS)]])
    r, v = R@p[:3], R@p[3:]
    rn = np.linalg.norm(r)
    h = np.cross(r, v)
    hn = np.linalg.norm(h)
    ev = np.cross(v, h)/MU - r/rn
    e = np.linalg.norm(ev)
    a = 1/(2/rn - v@v/MU)
    inc = np.degrees(np.arccos(h[2]/hn))
    n = np.cross([0,0,1], h)
    nn = np.linalg.norm(n)
    Om = np.degrees(np.arctan2(n[1], n[0])) % 360
    w = np.degrees(np.arccos(np.clip(n@ev/(nn*e), -1, 1)))
    w = w if ev[2] >= 0 else 360-w
    nu = np.arccos(np.clip(ev@r/(e*rn), -1, 1))
    nu = nu if r@v >= 0 else 2*np.pi-nu
    E = 2*np.arctan2(np.sqrt(1-e)*np.sin(nu/2), np.sqrt(1+e)*np.cos(nu/2))
    M = np.degrees(E - e*np.sin(E)) % 360
    return dict(a=a, e=e, i=inc, Om=Om, w=w, M=M, P_yr=2*np.pi*np.sqrt(abs(a)**3/MU)/365.25)

def initial_guess(jd, ra, dec, Robs, jd_ref):
    """Angles-only start: fit RA/Dec(t) with a quadratic, then grid over (rho, rho_dot)."""
    t = jd - jd_ref
    cr = np.polyfit(t, np.unwrap(ra), 2)
    cd = np.polyfit(t, dec, 2)
    a0, d0 = cr[2], cd[2]
    ad, dd = cr[1], cd[1]
    u_ = np.array([np.cos(d0)*np.cos(a0), np.cos(d0)*np.sin(a0), np.sin(d0)])
    ua = np.array([-np.cos(d0)*np.sin(a0), np.cos(d0)*np.cos(a0), 0.0])
    ud = np.array([-np.sin(d0)*np.cos(a0), -np.sin(d0)*np.sin(a0), np.cos(d0)])
    R0 = observer_helio(jd_ref)
    dt = 1e-3
    Rv = (observer_helio(jd_ref+dt) - observer_helio(jd_ref-dt))/(2*dt)
    best = None
    for rho in np.linspace(0.8, 5.0, 43):
        for rd in np.linspace(-0.03, 0.03, 31):
            r = R0 + rho*u_
            v = Rv + rd*u_ + rho*(ad*np.cos(d0)*ua + dd*ud)
            en = 0.5*v@v - K**2/np.linalg.norm(r)
            if en >= 0: continue
            p = np.concatenate([r, v])
            res = residuals(p, jd, jd_ref, Robs, ra, dec, np.radians(1/3600))
            c = res@res
            if best is None or c < best[0]: best = (c, p)
    return best[1]

def fit_orbit(jd, ra_deg, dec_deg, sig_arcsec):
    jd = np.asarray(jd); ra = np.radians(ra_deg); dec = np.radians(dec_deg)
    sig = np.radians(np.broadcast_to(sig_arcsec, jd.shape)/3600)
    sig2 = np.concatenate([sig, sig])
    Robs = np.array([observer_helio(j) for j in jd])
    jd_ref = float(np.mean(jd))
    p0 = initial_guess(jd, ra, dec, Robs, jd_ref)
    f = lambda p: residuals(p, jd, jd_ref, Robs, ra, dec, sig2[:len(jd)] if False else sig)
    sol = least_squares(f, p0, x_scale=np.abs(p0)+1e-3, xtol=1e-14, ftol=1e-14, gtol=1e-14)
    J = sol.jac; cov = np.linalg.pinv(J.T@J)
    dof = max(len(jd)*2 - 6, 1); chi2 = (sol.fun**2).sum()
    rms = np.sqrt(np.mean((sol.fun*np.tile(np.degrees(sig)*3600, 2))**2))
    return dict(p=sol.x, cov=cov, jd_ref=jd_ref, chi2=chi2, dof=dof, rms_arcsec=rms,
                resid=sol.fun*np.tile(np.degrees(sig)*3600, 2), Robs=Robs, p0=p0)

# ---------------------------------------------------------------- extras
A_JUP = 5.2044   # Jupiter semi-major axis [AU]

def tisserand_jupiter(a, e, i_deg):
    """Tisserand parameter w.r.t. Jupiter (the 'Jupiter Tisserand invariant')."""
    return A_JUP/a + 2*np.cos(np.radians(i_deg))*np.sqrt(a/A_JUP*(1-e**2))

def orbit_points(a, e, i_deg, Om_deg, w_deg, n=600):
    """Points along an orbit (ecliptic J2000 x,y,z in AU) from its elements."""
    nu = np.linspace(0, 2*np.pi, n)
    r = a*(1-e**2)/(1+e*np.cos(nu))
    x, y = r*np.cos(nu), r*np.sin(nu)                         # orbital plane
    i, Om, w = np.radians(i_deg), np.radians(Om_deg), np.radians(w_deg)
    cw, sw, cO, sO, ci, si = np.cos(w), np.sin(w), np.cos(Om), np.sin(Om), np.cos(i), np.sin(i)
    X = (cO*cw - sO*sw*ci)*x + (-cO*sw - sO*cw*ci)*y
    Y = (sO*cw + cO*sw*ci)*x + (-sO*sw + cO*cw*ci)*y
    Z = (sw*si)*x + (cw*si)*y
    return np.array([X, Y, Z])

def eq_to_ecl(v):
    R = np.array([[1,0,0],[0,np.cos(EPS),np.sin(EPS)],[0,-np.sin(EPS),np.cos(EPS)]])
    return R @ np.asarray(v).T

## 02 Orbit Fit

In [7]:
"""
Step 2: fit a two-body orbit to astrometry_table.csv (from 01_astrometry.py).
Needs orbit_lib.py in the same folder. Only YOUR measurements go into the fit;
JPL SBDB is looked up afterwards, for comparison only.
Outputs: orbit_elements.csv, orbit_result.npz (used by 03_orbit_plot.py), residuals.png
"""
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from astropy.visualization import ImageNormalize, ZScaleInterval, LinearStretch
from reproject import reproject_interp

tab = pd.read_csv("astrometry_table.csv")
jd, ra, dec, sig = tab.jd_mid.values, tab.ra_deg.values, tab.dec_deg.values, tab.sigma_arcsec.values
n = len(jd)

res = fit_orbit(jd, ra, dec, sig)
el = elements(res['p'], res['jd_ref'])
el['q'] = el['a']*(1-el['e'])                                   # perihelion distance
el['TJ'] = tisserand_jupiter(el['a'], el['e'], el['i'])         # Jupiter Tisserand invariant
print(f"N obs = {n}  ({2*n} measurements, 6 fitted parameters)")
print(f"reduced chi2 = {res['chi2']/res['dof']:.2f}   RMS residual = {res['rms_arcsec']:.2f} arcsec")
print(f"orbital epoch (JD UTC) = {res['jd_ref']:.4f}")

# ---- uncertainties: draw 3000 orbits consistent with the data, convert each to elements
rng = np.random.default_rng(1)
scale = max(1.0, np.sqrt(res['chi2']/res['dof']))
samp = rng.multivariate_normal(res['p'], res['cov']*scale**2, 3000)
keys = ['a', 'e', 'q', 'i', 'P_yr', 'TJ', 'Om', 'w', 'M']
S = {k: [] for k in keys}
for p in samp:
    try:
        d = elements(p, res['jd_ref'])
    except Exception:
        continue
    if d['a'] > 0 and 0 <= d['e'] < 1:
        d['q'] = d['a']*(1-d['e']); d['TJ'] = tisserand_jupiter(d['a'], d['e'], d['i'])
        for k in keys: S[k].append(d[k])

# ---- JPL SBDB (comparison only)
sb = {}
try:
    from astroquery.jplsbdb import SBDB
    o = SBDB.query('248370')['orbit']['elements']
    f = lambda x: float(getattr(x, 'value', x))
    sb = dict(a=f(o['a']), e=f(o['e']), i=f(o['i']), q=f(o['q']))
    sb['P_yr'] = sb['a']**1.5
    sb['TJ'] = tisserand_jupiter(sb['a'], sb['e'], sb['i'])
    sb.update(Om=f(o['om']), w=f(o['w']), M=f(o['ma']))
except Exception as ex:
    print("(SBDB lookup skipped:", ex, ")")

units = dict(a='AU', e='', q='AU', i='deg', P_yr='yr', TJ='', Om='deg', w='deg', M='deg')
names = dict(a='semi-major axis a', e='eccentricity e', q='perihelion q', i='inclination i',
             P_yr='period P', TJ='Tisserand T_J(Jupiter)', Om='long. asc. node Omega',
             w='arg. perihelion omega', M='mean anomaly M')
rows = []
print(f"\n{'parameter':26s}{'best fit':>11s}{'  +/- (1 sigma)':>16s}{'   SBDB':>10s}{'  diff/sigma':>12s}")
for k in keys:
    lo, md, hi = np.percentile(S[k], [16, 50, 84])
    sg = 0.5*(hi - lo)
    s = sb.get(k, np.nan)
    dev = (el[k] - s)/sg if np.isfinite(s) else np.nan
    print(f"{names[k]:26s}{el[k]:11.4f}{sg:16.4f}{s:10.4f}{dev:12.1f}  {units[k]}")
    rows.append(dict(parameter=k, unit=units[k], best_fit=el[k], sigma=sg, p16=lo, median=md, p84=hi, sbdb=s))
pd.DataFrame(rows).to_csv("orbit_elements.csv", index=False)
print("\n(angles: ecliptic J2000, at the orbital epoch above; SBDB's angles are at ITS epoch, so compare a, e, q, i, P, T_J)")
print(f"valid samples: {len(S['a'])} of 3000")

np.savez("orbit_result.npz", p=res['p'], jd_ref=res['jd_ref'], sb=np.array([sb.get(k, np.nan) for k in ['a','e','i','Om','w']]))

r = res['resid']; dra, ddec = r[:n], r[n:]
fig, ax = plt.subplots(1, 2, figsize=(10, 4)); t = jd - jd.min()
ax[0].errorbar(t, dra, sig, fmt='o'); ax[0].axhline(0, c='k', lw=.5)
ax[0].set_title("RA*cos(Dec) residual (model - obs)"); ax[0].set_xlabel("days since first frame"); ax[0].set_ylabel("arcsec")
ax[1].errorbar(t, ddec, sig, fmt='o'); ax[1].axhline(0, c='k', lw=.5)
ax[1].set_title("Dec residual (model - obs)"); ax[1].set_xlabel("days since first frame")
plt.tight_layout(); plt.savefig("residuals.png", dpi=150)

N obs = 15  (30 measurements, 6 fitted parameters)
reduced chi2 = 0.40   RMS residual = 0.23 arcsec
orbital epoch (JD UTC) = 2459520.0038

parameter                    best fit   +/- (1 sigma)      SBDB  diff/sigma
semi-major axis a              3.0631          0.0065    3.0700        -1.1  AU
eccentricity e                 0.2271          0.0026    0.2250         0.8  
perihelion q                   2.3675          0.0130    2.3800        -1.0  AU
inclination i                  0.0669          0.0007    0.0681        -1.8  deg
period P                       5.3610          0.0171    5.3791        -1.1  yr
Tisserand T_J(Jupiter)         3.1933          0.0011    3.1919         1.2  
long. asc. node Omega        174.1534          0.1497  174.0000         1.0  deg
arg. perihelion omega        145.2694          1.9086  146.0000        -0.4  deg
mean anomaly M                32.0493          1.1355  340.0000      -271.2  deg

(angles: ecliptic J2000, at the orbital epoch above; SBDB's angl

## 03 Orbit Plot

In [ ]:
"""
Step 3: 3-D picture of the Sun, Earth (+orbit), the asteroid (+orbit from YOUR fit)
and, for comparison, the orbit from JPL SBDB's published elements.
Run after 02_orbit_fit.py. Saves orbit_3d.png.
"""
from astropy.coordinates import get_body_barycentric, solar_system_ephemeris

solar_system_ephemeris.set('builtin')
R = np.load("orbit_result.npz")
p, jd_ref = R['p'], float(R['jd_ref'])

# ---- your fitted orbit
el = elements(p, jd_ref)
mine = orbit_points(el['a'], el['e'], el['i'], el['Om'], el['w'])
ast_now = eq_to_ecl(p[:3])                      # asteroid position at the orbital epoch (ecliptic, AU)

# ---- JPL SBDB orbit (a, e, i, Omega, omega). If the lookup failed in step 2, type the values here:
a_s, e_s, i_s, Om_s, w_s = R['sb']
MANUAL = None            # e.g. MANUAL = (3.07, 0.225, 0.0681, 174.0, 145.0)  <- from SBDB / Wikipedia
if MANUAL: a_s, e_s, i_s, Om_s, w_s = MANUAL
sbdb = orbit_points(a_s, e_s, i_s, Om_s, w_s) if np.all(np.isfinite([a_s, e_s, i_s, Om_s, w_s])) else None

# ---- Earth: real positions over one year from the ephemeris
t = Time(jd_ref + np.linspace(0, 365.25, 400), format='jd', scale='utc')
earth = eq_to_ecl(((get_body_barycentric('earth', t) - get_body_barycentric('sun', t)).xyz.to(u.au).value).T)
t0 = Time(jd_ref, format='jd', scale='utc')
earth_now = eq_to_ecl((get_body_barycentric('earth', t0) - get_body_barycentric('sun', t0)).xyz.to(u.au).value)

fig = plt.figure(figsize=(14, 6.5))
ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.plot(*earth, c='tab:blue', lw=1.5, label='Earth orbit')
ax.plot(*mine, c='tab:red', lw=1.5, label=f"asteroid orbit (our fit): a={el['a']:.2f}, e={el['e']:.3f}")
if sbdb is not None: ax.plot(*sbdb, c='k', ls='--', lw=1.2, label='JPL SBDB orbit')
ax.scatter(0, 0, 0, c='gold', s=200, edgecolors='orange', label='Sun')
ax.scatter(*earth_now, c='tab:blue', s=40, label='Earth (at epoch)')
ax.scatter(*ast_now, c='tab:red', s=40, marker='D', label='2005 QN173 (at epoch)')
ax.set_xlabel('x [AU]'); ax.set_ylabel('y [AU]'); ax.set_zlabel('z [AU]')
lim = 3.8; ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
ax.set_box_aspect((1, 1, 1)); ax.view_init(elev=28, azim=-60)
ax.set_title("Ecliptic J2000, heliocentric (drag to rotate if run interactively)")
ax.legend(loc='upper left', fontsize=8)

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(earth[0], earth[1], c='tab:blue'); ax2.plot(mine[0], mine[1], c='tab:red')
if sbdb is not None: ax2.plot(sbdb[0], sbdb[1], c='k', ls='--')
ax2.scatter(0, 0, c='gold', s=200, edgecolors='orange')
ax2.scatter(*earth_now[:2], c='tab:blue', s=40); ax2.scatter(*ast_now[:2], c='tab:red', marker='D', s=40)
ax2.set_aspect('equal'); ax2.set_xlabel('x [AU]'); ax2.set_ylabel('y [AU]')
ax2.set_title(f"Top-down view (i = {el['i']:.2f} deg, so the orbit is nearly flat)"); ax2.grid(alpha=.3)
plt.tight_layout(); plt.savefig("orbit_3d.png", dpi=150)
print("saved orbit_3d.png")
plt.show()

saved orbit_3d.png


## 04 Interactive Frame Stepper & Dual Animations (Sidereal vs. Asteroid-Locked)

In [ ]:

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not installed; skipping interactive button UI (GIFs will still be generated).")

# -------------------------------------------------------------------------
# 1. Load astrometry measurements (produced by Step 1)
# -------------------------------------------------------------------------
tab = pd.read_csv("astrometry_table.csv")
tab["jd_mid"] = pd.to_numeric(tab["jd_mid"])
tab = tab.sort_values("jd_mid").reset_index(drop=True)

files = tab["file"].tolist()
ra_deg = tab["ra_deg"].values
dec_deg = tab["dec_deg"].values
tmid_utc = tab["tmid_utc"].values
jds = tab["jd_mid"].values
n_frames = len(files)

# -------------------------------------------------------------------------
# 2. Build common sidereal reference WCS centered on the asteroid's arc
# -------------------------------------------------------------------------
ra_center = float(np.mean(ra_deg))
dec_center = float(np.mean(dec_deg))
COMMON_SIZE = (1800, 1800)       # (ny, nx)
PIXEL_SCALE_DEG = 0.675 / 3600.0 # ~0.675 arcsec/px

ref_hdr = fits.Header()
ref_hdr["NAXIS"] = 2
ref_hdr["NAXIS1"] = COMMON_SIZE[1]
ref_hdr["NAXIS2"] = COMMON_SIZE[0]
ref_hdr["CTYPE1"] = "RA---TAN"
ref_hdr["CTYPE2"] = "DEC--TAN"
ref_hdr["CRVAL1"] = ra_center
ref_hdr["CRVAL2"] = dec_center
ref_hdr["CRPIX1"] = COMMON_SIZE[1] / 2.0
ref_hdr["CRPIX2"] = COMMON_SIZE[0] / 2.0
ref_hdr["CDELT1"] = -PIXEL_SCALE_DEG
ref_hdr["CDELT2"] = PIXEL_SCALE_DEG

common_wcs = WCS(ref_hdr)
ast_x_ref, ast_y_ref = common_wcs.all_world2pix(ra_deg, dec_deg, 0)

# -------------------------------------------------------------------------
# 3. Reproject all frames onto stationary star grid with clean contrast
# -------------------------------------------------------------------------
sidereal_frames = []
for idx, f in enumerate(files):
    with fits.open(f) as hdul:
        data = hdul[0].data.astype(np.float32)
        wcs_in = WCS(hdul[0].header)

    aligned, footprint = reproject_interp((data, wcs_in), common_wcs, shape_out=COMMON_SIZE)
    valid_mask = footprint > 0.5
    valid_pixels = aligned[valid_mask]

    vmin, vmax = ZScaleInterval(contrast=0.08).get_limits(valid_pixels)
    norm = ImageNormalize(vmin=vmin, vmax=vmax, stretch=LinearStretch(), clip=True)

    img = norm(aligned)
    img[~valid_mask] = 0.0
    sidereal_frames.append(img)
    print(f"[{idx+1:02d}/{n_frames}] Reprojected {f}")

sidereal_frames = np.array(sidereal_frames)

# Bounding box around the asteroid's full observed path (for stationary-star view)
MARGIN = 120
x_min_s, x_max_s = ast_x_ref.min() - MARGIN, ast_x_ref.max() + MARGIN
y_min_s, y_max_s = ast_y_ref.min() - MARGIN, ast_y_ref.max() + MARGIN

# -------------------------------------------------------------------------
# 4. Interactive Jupyter Stepper (Back / Next / Slider / Play)
# -------------------------------------------------------------------------
if HAS_WIDGETS:
    out_widget = widgets.Output()
    slider = widgets.IntSlider(
        value=0, min=0, max=n_frames - 1, step=1,
        description="Frame:", continuous_update=False
    )
    btn_prev = widgets.Button(description="◀ Back", layout=widgets.Layout(width="90px"))
    btn_next = widgets.Button(description="Next ▶", layout=widgets.Layout(width="90px"))
    play = widgets.Play(
        interval=800, value=0, min=0, max=n_frames - 1, step=1,
        description="Play", disabled=False
    )
    widgets.jslink((play, "value"), (slider, "value"))

    def render_interactive_frame(idx):
        with out_widget:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(8.5, 8.5), facecolor="black")
            plt.subplots_adjust(left=0.02, right=0.98, bottom=0.05, top=0.93)

            ax.imshow(sidereal_frames[idx], cmap="gray", origin="lower", vmin=0, vmax=1)
            ax.plot(ast_x_ref[:idx+1], ast_y_ref[:idx+1], "c--", lw=1.3, alpha=0.7, label="Observed Path")
            ax.scatter(ast_x_ref[:idx], ast_y_ref[:idx], c="cyan", s=20, alpha=0.55, edgecolors="none")
            ax.plot([ast_x_ref[idx]], [ast_y_ref[idx]], "ro", markersize=8,
                    fillstyle="none", markeredgewidth=2.0, label="2005 QN173")

            ax.set_xlim(x_min_s, x_max_s)
            ax.set_ylim(y_min_s, y_max_s)
            ax.axis("off")

            elapsed = jds[idx] - jds[0]
            t_str = str(tmid_utc[idx])[:16].replace("T", " ")
            ax.set_title(
                f"Frame {idx+1}/{n_frames} | UTC: {t_str} | Day +{elapsed:.2f}d | "
                f"RA={ra_deg[idx]:.4f}°, Dec={dec_deg[idx]:.4f}°",
                color="yellow", fontsize=10, pad=8
            )
            ax.legend(loc="upper right", facecolor="black", labelcolor="white", framealpha=0.65, fontsize=9)
            plt.show()
            plt.close(fig)

    def on_prev(_):
        slider.value = (slider.value - 1) % n_frames

    def on_next(_):
        slider.value = (slider.value + 1) % n_frames

    def on_slider_change(change):
        render_interactive_frame(change["new"])

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    slider.observe(on_slider_change, names="value")

    controls = widgets.HBox([btn_prev, btn_next, play, slider])
    display(controls, out_widget)
    render_interactive_frame(0)

# -------------------------------------------------------------------------
# 5. GIF 1: Stationary Stars, Moving Asteroid
# -------------------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(8, 8), facecolor="black")
plt.subplots_adjust(left=0.02, right=0.98, bottom=0.04, top=0.93)

im1 = ax1.imshow(sidereal_frames[0], cmap="gray", origin="lower", vmin=0, vmax=1)
(marker1,) = ax1.plot([], [], "ro", markersize=7, fillstyle="none", markeredgewidth=1.8, label="2005 QN173")
(trail1,) = ax1.plot([], [], "c--", lw=1.2, alpha=0.65, label="Observed Path")
hud1 = ax1.text(
    0.03, 0.95, "", transform=ax1.transAxes, color="yellow", fontsize=10, weight="bold",
    bbox=dict(boxstyle="round,pad=0.35", fc="black", ec="yellow", alpha=0.75)
)

ax1.set_xlim(x_min_s, x_max_s)
ax1.set_ylim(y_min_s, y_max_s)
ax1.axis("off")
ax1.legend(loc="upper right", facecolor="black", labelcolor="white", framealpha=0.65, fontsize=8.5)

def update_gif1(i):
    im1.set_data(sidereal_frames[i])
    marker1.set_data([ast_x_ref[i]], [ast_y_ref[i]])
    trail1.set_data(ast_x_ref[:i+1], ast_y_ref[:i+1])
    elapsed = jds[i] - jds[0]
    t_str = str(tmid_utc[i])[:16].replace("T", " ")
    hud1.set_text(f"Sidereal View | {t_str} UTC | Day +{elapsed:.2f} ({i+1}/{n_frames})")
    return im1, marker1, trail1, hud1

ani1 = animation.FuncAnimation(fig1, update_gif1, frames=n_frames, interval=800, blit=False)
ani1.save("asteroid_stationary_stars.gif", writer="pillow", fps=1.25, dpi=120)
plt.close(fig1)
print("Saved: asteroid_stationary_stars.gif")

# -------------------------------------------------------------------------
# 6. GIF 2: Asteroid Locked in Center, Stars Moving Around It
# -------------------------------------------------------------------------
# Crop a window around the asteroid's position in each frame so the asteroid
# stays fixed at the center while background stars shift across the field.
HALF_BOX = 160  # +/- 160 px (~108 arcsec half-width) around asteroid

locked_frames = []
for i in range(n_frames):
    cx, cy = ast_x_ref[i], ast_y_ref[i]
    # Sample a (2*HALF_BOX) x (2*HALF_BOX) patch centered at (cx, cy)
    x_grid = np.arange(cx - HALF_BOX, cx + HALF_BOX)
    y_grid = np.arange(cy - HALF_BOX, cy + HALF_BOX)
    xx, yy = np.meshgrid(x_grid, y_grid)

    ix = np.clip(np.rint(xx).astype(int), 0, COMMON_SIZE[1] - 1)
    iy = np.clip(np.rint(yy).astype(int), 0, COMMON_SIZE[0] - 1)
    in_bounds = (xx >= 0) & (xx < COMMON_SIZE[1]) & (yy >= 0) & (yy < COMMON_SIZE[0])

    patch = np.where(in_bounds, sidereal_frames[i, iy, ix], 0.0)
    locked_frames.append(patch)

locked_frames = np.array(locked_frames)

fig2, ax2 = plt.subplots(figsize=(8, 8), facecolor="black")
plt.subplots_adjust(left=0.02, right=0.98, bottom=0.04, top=0.93)

im2 = ax2.imshow(locked_frames[0], cmap="gray", origin="lower", vmin=0, vmax=1)
# Reticle / ring at center where the asteroid is locked
ax2.plot([HALF_BOX], [HALF_BOX], "ro", markersize=9, fillstyle="none", markeredgewidth=1.8, label="Locked: 2005 QN173")
hud2 = ax2.text(
    0.03, 0.95, "", transform=ax2.transAxes, color="yellow", fontsize=10, weight="bold",
    bbox=dict(boxstyle="round,pad=0.35", fc="black", ec="yellow", alpha=0.75)
)

ax2.set_xlim(0, 2 * HALF_BOX)
ax2.set_ylim(0, 2 * HALF_BOX)
ax2.axis("off")
ax2.legend(loc="upper right", facecolor="black", labelcolor="white", framealpha=0.65, fontsize=8.5)

def update_gif2(i):
    im2.set_data(locked_frames[i])
    elapsed = jds[i] - jds[0]
    t_str = str(tmid_utc[i])[:16].replace("T", " ")
    hud2.set_text(f"Asteroid-Locked | {t_str} UTC | Day +{elapsed:.2f} ({i+1}/{n_frames})")
    return im2, hud2

ani2 = animation.FuncAnimation(fig2, update_gif2, frames=n_frames, interval=800, blit=False)
ani2.save("asteroid_locked_moving_stars.gif", writer="pillow", fps=1.25, dpi=120)
plt.close(fig2)
print("Saved: asteroid_locked_moving_stars.gif")

[01/15] Reprojected 20211022181158-894-RA.wcs.proc.fits.gz
[02/15] Reprojected 20211022182027-424-RA.wcs.proc.fits.gz
[03/15] Reprojected 20211026184308-020-RA.wcs.proc.fits.gz
[04/15] Reprojected 20211026184717-057-RA.wcs.proc.fits.gz
[05/15] Reprojected 20211029182022-389-RA.wcs.proc.fits.gz
[06/15] Reprojected 20211029182431-429-RA.wcs.proc.fits.gz
[07/15] Reprojected 20211031190017-266-RA.wcs.proc.fits.gz
[08/15] Reprojected 20211031190426-339-RA.wcs.proc.fits.gz
[09/15] Reprojected 20211103182047-952-RA.wcs.proc.fits.gz
[10/15] Reprojected 20211103182457-013-RA.wcs.proc.fits.gz
[11/15] Reprojected 20211106181632-807-RA.wcs.proc.fits.gz
[12/15] Reprojected 20211106182057-122-RA.wcs.proc.fits.gz
[13/15] Reprojected 20211107180417-493-RA.wcs.proc.fits.gz
